In [3]:
import pandas as pd
import numpy as np
import pywt
import plotly.graph_objects as go
from scipy.signal import find_peaks

class Zigzag:
    def __init__(self, timeseries, swingthresh=None):
        self.timeseries = timeseries
        self.swingthresh = swingthresh

    def get_swings(self):
        prices = self.timeseries['close']
        peak_indices, _ = find_peaks(prices, prominence=prices.max()*0.001, threshold=0.001)
        trough_indices, _ = find_peaks(-prices, prominence=prices.max()*0.001, threshold=0.001)

        # Sort peaks and troughs into a single ordered array
        extrema_indices = sorted(np.concatenate((peak_indices, trough_indices)))

        swings = []
        last_extremum_price = prices.iloc[extrema_indices[0]]
        last_extremum_type = 'high' if extrema_indices[0] in peak_indices else 'low'

        swings.append((prices.index[extrema_indices[0]], last_extremum_price, last_extremum_type))

        for idx in extrema_indices[1:]:
            price = prices.iloc[idx]
            current_extremum_type = 'high' if idx in peak_indices else 'low'
            change = abs((price - last_extremum_price)) / last_extremum_price

            # Check if the current extremum is different from the last and meets the threshold
            if current_extremum_type != last_extremum_type:
                if change >= self.swingthresh:
                    swings.append((prices.index[idx], price, current_extremum_type))
                    last_extremum_price = price
                    last_extremum_type = current_extremum_type
            else:  # Handle consecutive maxima or minima
                if (current_extremum_type == 'high' and price > last_extremum_price) or \
                   (current_extremum_type == 'low' and price < last_extremum_price):
                    swings[-1] = (prices.index[idx], price, current_extremum_type)
                    last_extremum_price = price

        return swings

data = pd.read_csv('E:\SignalModel\one-minute data in one month\price 2024-02-01, 2024-03-01.csv')
data['datetime'] = pd.to_datetime(data['datetime'])
data.set_index('datetime', inplace=True)
data.sort_index(inplace=True)
data['close'] = data['close'].astype(float)

# Initialize Zigzag with the swing threshold and get the swings
zigzag_indicator = Zigzag(data, swingthresh=0.005)
swings = zigzag_indicator.get_swings()

# Prepare the data for plotting
swing_times = [swing[0] for swing in swings]
swing_prices = [swing[1] for swing in swings]
swing_types = [swing[2] for swing in swings]

# Plotting
fig = go.Figure()

# Add trace for close prices
fig.add_trace(go.Scatter(x=data.index, y=data['close'], mode='lines', name='Close Price'))

# Plot the swings: connect them with a line
for i in range(len(swing_times) - 1):
    fig.add_trace(go.Scatter(
        x=[swing_times[i], swing_times[i+1]], 
        y=[swing_prices[i], swing_prices[i+1]], 
        mode='lines+markers', 
        name=f'Swing {i}',
        line=dict(color='red' if swing_types[i] == 'high' else 'green', width=2),
        marker=dict(color='red' if swing_types[i] == 'high' else 'green', size=8)
    ))

# Update layout for readability
fig.update_layout(title='BTC Close Price with local max and min within 0.005 percentage change', xaxis_title='Date', yaxis_title='Price', showlegend=False)

# Show plot
fig.show()
